# 01 - Ingesta de datos crudos en AWS S3

Este notebook tiene como objetivo automatizar la descarga de datos públicos de viajes (NYC TLC Trip Record Data) y su posterior almacenamiento en un bucket de Amazon S3 de forma particionada.

**Dependencias y configuración inicial:**
* `boto3`: Cliente de AWS para interactuar con S3. Requiere que las credenciales de AWS estén configuradas en el entorno.
* `requests`: Para realizar peticiones HTTP y descargar los archivos `.parquet`.

En este bloque definimos el bucket de destino (`xideralaws-curso-proyecto-alan`) y establecemos la conexión con la región `us-west-1`.

In [1]:
import boto3
import requests
BUCKET = "xideralaws-curso-proyecto-alan"

In [2]:
# Inicializamos el cliente
s3 = boto3.client("s3", region_name="us-west-1")

## Función de extracción y carga

La función `upload_tlc_data_to_s3` encapsula la lógica para mover cada archivo individual desde el repositorio público hasta nuestro bucket en S3. 

**Características principales:**
1. **Generación dinámica de rutas:** Construye la URL de origen y la llave de destino en S3 (`s3_key`).
2. **Particionamiento:** Organiza los datos en S3 usando una jerarquía de estilo data lake (`raw_data/YYYY/MM/`), lo cual facilitará futuras consultas con herramientas como AWS Athena.
3. **Optimización de memoria:** Utiliza `stream=True` en la petición `requests.get` junto con `upload_fileobj`. Esto transfiere los datos en trozos directamente a S3 sin cargar todo el archivo `.parquet` en la memoria RAM del entorno local.

*Retorna `True` si el archivo se subió con éxito (código HTTP 200) o `False` si el archivo no existe (ej. meses futuros).*

In [3]:
def upload_tlc_data_to_s3(year, month, taxi_type="yellow"):
    month_str = f"{month:02d}"
    # URL de ejemplo: #https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet
    file_name = f"{taxi_type}_tripdata_{year}-{month_str}.parquet"
    
    # URL oficial del repositorio de NYC TLC
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file_name}"
    
    # Utilizaremos una jerarquia de anos y meses para mantener un orden mas estricto
    s3_key = f"raw_data/{year}/{month_str}/{file_name}"
    
    # stream=True evita descargar todo el archivo en la memoria local
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        print(f"Subiendo: {file_name} -> s3://{BUCKET}/{s3_key}")
        # Sube directamente el flujo de descarga al bucket S3
        s3.upload_fileobj(response.raw, BUCKET, s3_key)
        return True
    else:
        return False

## Ejecución del pipeline de ingesta

En este bloque se define el alcance temporal y los tipos de datos a ingerir, y se ejecuta el ciclo de descarga.

**Parámetros:**
* **Años a procesar:** 2024, 2025 y 2026.
* **Tipos de transporte:** `yellow`, `green`, `fhv` y `fhvhv`.

**Lógica de validación:**
El script itera mes a mes probando descargar todos los tipos de taxi. Si un mes entero falla para todos los tipos (por ejemplo, mayo de 2026 en adelante, ya que los datos aún no se han publicado), la variable `datos_encontrados_en_mes` permanece en `False`. Esto activa un `break` que detiene las peticiones innecesarias para el resto de ese año, ahorrando tiempo de ejecución y ancho de banda.

In [4]:
# Parámetros de descarga
years = [2024, 2025, 2026]
taxi_types = ["yellow", "green", "fhv", "fhvhv"]

for year in years:
    print(f"\n--- Iniciando ingesta del año {year} ---")
    for month in range(1, 13):
        datos_encontrados_en_mes = False
        
        for t_type in taxi_types:
            # Si la descarga es exitosa, marcamos que el mes sí tiene datos
            if upload_tlc_data_to_s3(year, month, t_type):
                datos_encontrados_en_mes = True
        
        # Validar meses incompletos (ej. 2026 solo tiene hasta abril)
        if not datos_encontrados_en_mes:
            print(f"No hay más datos disponibles a partir del mes {month:02d} de {year}. Saltando al siguiente año.")
            break 

print("\n¡Ingesta y partición de datos en S3 completada con éxito!")


--- Iniciando ingesta del año 2024 ---
Subiendo: yellow_tripdata_2024-01.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/01/yellow_tripdata_2024-01.parquet
Subiendo: green_tripdata_2024-01.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/01/green_tripdata_2024-01.parquet
Subiendo: fhv_tripdata_2024-01.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/01/fhv_tripdata_2024-01.parquet
Subiendo: fhvhv_tripdata_2024-01.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/01/fhvhv_tripdata_2024-01.parquet
Subiendo: yellow_tripdata_2024-02.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/02/yellow_tripdata_2024-02.parquet
Subiendo: green_tripdata_2024-02.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/02/green_tripdata_2024-02.parquet
Subiendo: fhv_tripdata_2024-02.parquet -> s3://xideralaws-curso-proyecto-alan/raw_data/2024/02/fhv_tripdata_2024-02.parquet
Subiendo: fhvhv_tripdata_2024-02.parquet -> s3://xideralaws-curso-pr